# Linux Bash: Startup Files, History, Redirection, Job Control, and Options (Educational Notebook)
This notebook builds on `Lesson3-Linux_Shell_and_Bash_Fundamentals.ipynb` and covers how Bash starts up, how to reuse and control commands, how to redirect input/output, and how to run and manage multiple jobs at once.

## 1. Bash Startup Files

Bash reads different configuration files depending on whether it is started as a **login shell** (e.g. a fresh terminal login, or `ssh` connecting in) or a **non-login (interactive) shell** (e.g. a new terminal tab in an already-logged-in desktop session), and whether it's interactive or running a script.

| File | Read by |
|---|---|
| `/etc/profile` | Login shells (system-wide) |
| `~/.bash_profile` (or `~/.profile`) | Login shells (per-user), often just sources `~/.bashrc` |
| `/etc/bash.bashrc` | Interactive non-login shells (system-wide, distro-dependent) |
| `~/.bashrc` | Interactive non-login shells (per-user) - the usual place for aliases, functions, and `PATH` tweaks |
| `~/.bash_logout` | Run when a login shell exits |

Because the exact rules are easy to get wrong, a common, safe convention is to put persistent customizations in `~/.bashrc`, and have `~/.bash_profile` simply `source ~/.bashrc` so login shells pick them up too.

```bash
source ~/.bashrc      # re-read a startup file into the current shell without logging out/in
. ~/.bashrc              # "." is a shorthand builtin equivalent to "source"
```


## 2. Command History

Bash keeps a record of previously entered commands, letting you recall and reuse them instead of retyping.

```bash
history                  # list recent commands with their history number
history 20                  # list just the last 20
!42                            # re-run history entry number 42
!!                                # re-run the previous command
!ping                              # re-run the most recent command starting with "ping"
!$                                   # reuse the last argument of the previous command
```

Interactively, `Ctrl+R` starts a **reverse incremental search** through history - type part of a remembered command and Bash finds the most recent match; press `Ctrl+R` again to search further back.

History is controlled by environment variables such as `HISTSIZE` (commands kept in memory for the session) and `HISTFILESIZE` (commands kept in the history file, `~/.bash_history`, across sessions).

```bash
echo $HISTSIZE
export HISTSIZE=5000
```


## 3. Tab Completion

Pressing `Tab` asks Bash to complete whatever you've typed so far:

- Completing a **command name** searches builtins, functions, and `PATH`.
- Completing a **filename argument** searches the current (or specified) directory.
- Pressing `Tab` twice when the completion is ambiguous lists every possibility instead of completing.

Many distributions also install `bash-completion`, which adds smarter, command-aware completions (e.g. completing a `git` subcommand, or a running container's name for `docker exec`).


## 4. Standard Streams: stdin, stdout, stderr

Every process starts with three open communication channels, each identified by a small integer called a **file descriptor**:

| Stream | File descriptor | Default destination |
|---|---|---|
| **stdin** (standard input) | `0` | The keyboard |
| **stdout** (standard output) | `1` | The terminal screen |
| **stderr** (standard error) | `2` | The terminal screen |

Separating normal output (stdout) from error output (stderr) - even though both print to the screen by default - is what makes redirection useful: you can send a command's real output to a file while still seeing its errors on screen, or vice versa.


## 5. I/O Redirection

Redirection lets you connect a stream to a file instead of its default destination.

```bash
command > file        # redirect stdout to a file, overwriting it
command >> file          # redirect stdout to a file, appending
command 2> file            # redirect stderr to a file
command 2>> file             # append stderr to a file
command &> file                # redirect both stdout and stderr to the same file
command < file                   # use a file's contents as stdin instead of the keyboard

command > file 2>&1               # redirect stdout to a file, then point stderr at "wherever stdout now goes"
```

The order in `2>&1` matters: `command > file 2>&1` sends both streams to `file`, but `command 2>&1 > file` does not - it duplicates stderr to the *old* stdout (the terminal) *before* stdout gets redirected to the file, so stderr still prints to the screen.

```bash
ls /etc /nope > out.txt 2> err.txt      # split real output and error messages into two files
grep "pattern" < input.txt                # feed a file in as input rather than naming it as an argument
```


## 6. Pipes and Command Chaining

A **pipe** (`|`) connects one command's stdout directly to the next command's stdin, without an intermediate file - this is the classic Unix idea of building complex behavior from small, single-purpose tools.

```bash
ls -l | grep ".txt"              # feed ls's output into grep
ps aux | grep firefox               # find a running process by name
cat access.log | sort | uniq -c       # a small pipeline: sort lines, then count duplicates
```

Commands can also be **chained** based on their exit status (`0` means success, anything else means failure):

```bash
mkdir new_dir && cd new_dir      # run cd only if mkdir succeeded
grep "pattern" file || echo "not found"      # run echo only if grep failed (no match / error)
command1 ; command2                 # run command2 regardless of whether command1 succeeded
```


## 7. Job Control

Bash can run more than one command at a time from a single terminal, treating each as a **job**.

```bash
long_running_command &      # start a command in the background; "&" returns control immediately
jobs                          # list jobs running in this shell, with their job numbers
fg %1                           # bring job 1 to the foreground
bg %1                             # resume a stopped job 1 in the background
kill %1                             # send SIGTERM to job 1
```

While a command is running in the **foreground** (the normal case), pressing:
- `Ctrl+C` sends `SIGINT`, normally terminating it.
- `Ctrl+Z` sends `SIGTSTP`, **suspending** it (pausing, not killing) and returning you to the prompt - the paused job can then be resumed with `fg` or `bg`.

Closing the terminal normally sends `SIGHUP` to jobs still attached to it; tools like `nohup` or `disown` (or running the job inside `tmux`/`screen`) let a background job survive the terminal closing.


## 8. Shell Options

Bash's own behavior can be tuned with two related mechanisms:

```bash
set -o                # list all "set" options and their current state (on/off)
set -x                  # enable a specific option: here, print each command before executing it (useful for debugging scripts)
set +x                    # disable that option again
set -e                      # exit a script immediately if any command fails (common in scripts)

shopt                  # list all "shopt" options (a second, Bash-specific set of toggles) and their state
shopt -s globstar         # enable one: here, ** matches files recursively through subdirectories
shopt -u globstar           # disable it again
```

`set` options are POSIX-shell options shared with other shells (`sh`, `ksh`, ...); `shopt` options are Bash-specific extensions. Both are commonly set at the top of shell scripts to make their behavior stricter and more predictable, for example:

```bash
#!/bin/bash
set -euo pipefail
# -e: exit on any error
# -u: treat use of an unset variable as an error
# -o pipefail: a pipeline fails if *any* command in it fails, not just the last one
```


## Hands-on

Try these on your own system:

```bash
history | tail -5
echo $HISTSIZE
ls /etc /does_not_exist > out.txt 2> err.txt
cat out.txt
cat err.txt
ps aux | grep bash
sleep 100 &
jobs
fg
# press Ctrl+C to stop it, or let it finish
set -o | head -5
shopt | head -5
```


## Review Questions

1. What is the difference between a login shell and a non-login interactive shell, in terms of which startup files get read?
2. Why is it common to have `~/.bash_profile` simply source `~/.bashrc`?
3. What does `!!` do, and what does `!$` do?
4. Why does separating stdout and stderr into two streams matter, given both print to the screen by default?
5. Explain why `command 2>&1 > file` does *not* send stderr to `file`, while `command > file 2>&1` does.
6. What is the difference between what a pipe (`|`) does and what `&&` does?
7. What signal does `Ctrl+Z` send, and how is that different from `Ctrl+C`?
8. What does `set -e` do in a script, and why might a script author want that behavior?
9. What is the difference between a `set` option and a `shopt` option?
10. You start a long job in the background with `&`, then close the terminal. What normally happens to it, and name one way to prevent that.


# Cheat Sheet

```
Startup files:
  login shell:        /etc/profile -> ~/.bash_profile (or ~/.profile)
  non-login shell:      /etc/bash.bashrc -> ~/.bashrc
  source ~/.bashrc          re-read a startup file into the current shell

History:
  history | !42 | !! | !$ | Ctrl+R (reverse search)
  HISTSIZE, HISTFILESIZE, ~/.bash_history

Redirection:
  >   overwrite stdout        >>  append stdout
  2>  overwrite stderr         2>> append stderr
  &>  both streams               > file 2>&1   (both, in that order)
  <   read stdin from a file

Pipes & chaining:
  cmd1 | cmd2         pipe stdout of cmd1 into stdin of cmd2
  cmd1 && cmd2          run cmd2 only if cmd1 succeeded
  cmd1 || cmd2            run cmd2 only if cmd1 failed
  cmd1 ; cmd2                run both regardless

Job control:
  cmd &     jobs     fg %1     bg %1     kill %1
  Ctrl+C = SIGINT (terminate)     Ctrl+Z = SIGTSTP (suspend)

Options:
  set -o / set -x / set -e / set -euo pipefail
  shopt / shopt -s globstar
```
